# Lecture 2.8 — Tool Use Behaviour: `run_llm_again` vs `stop_on_first_tool`

**Course:** OpenAI Agents SDK — Complete Course  
**Section:** 02 — Agents: Configuration & Behaviour  
**Lecture:** 2.8 — Final lecture of Section 2

---

This notebook explores `tool_use_behavior` — the `Agent` parameter that controls what happens **after** a tool call completes. We work through all four valid values:

| Value | What it does |
|---|---|
| `"run_llm_again"` | Default. Tool result goes back to the LLM for a synthesised response. |
| `"stop_on_first_tool"` | Raw tool output becomes the final response. No second LLM call. |
| `StopAtTools(stop_at_tool_names=[...])` | Only named tools bypass the LLM. Others still go through it. |
| Custom `ToolsToFinalOutputFunction` | You inspect results and decide per-turn whether the run is done. |

## Cell 1 — Install the SDK

📌 **Notebook update notice:** the code shown in this lecture's video pins `openai-agents==0.17.4`. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version shown in the video. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one visible in the recording.

This cell installs `openai-agents`, pinned to a specific version for reproducibility. Every learner running this notebook will get the same API behaviour as the course examples.

To switch versions:
- **Latest release:** `pip install openai-agents`
- **Specific version:** change the version number below

If the package is already installed at this version in your current Colab session, `pip` will confirm it and skip reinstalling.

In [ ]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.17.4 as shown in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

## Cell 2 — API Key Setup

We load the OpenAI API key from **Google Colab Secrets** — your key is never written into the notebook.

### How to add your key in Colab

1. Click the **🔑 key icon** in the left sidebar (or go to *Tools → Secrets*).
2. Click **+ Add new secret**.
3. Set the **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key into the **Value** field.
5. Toggle **Notebook access** to **ON** for this notebook.
6. Run the cell below.

> **Running locally?** Set the environment variable in your terminal before starting Jupyter: `export OPENAI_API_KEY="sk-..."`

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Model Name

We declare `MODEL_NAME` once here and use it in every `Agent` definition throughout the notebook. Change this one variable to switch the model used across all cells.

We use `gpt-5.4-mini` — a fast, cost-efficient GPT-5 model well-suited for tool-calling demos.

> See the latest available models at: https://platform.openai.com/docs/models

In [ ]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

We import everything needed for this lecture.

| Import | From | Purpose |
|---|---|---|
| `Agent` | `agents` | Core agent class |
| `FunctionToolResult` | `agents` | Represents one tool call result — used in custom `ToolsToFinalOutputFunction` callbacks |
| `ModelSettings` | `agents` | Model tuning parameters (`reasoning`, `verbosity`, `tool_choice`, …) |
| `RunContextWrapper` | `agents` | Wraps the run context — first argument of any custom `tool_use_behavior` function |
| `Runner` | `agents` | Executes the agent run |
| `function_tool` | `agents` | Decorator that converts a Python function into an agent tool |
| `StopAtTools` | `agents.agent` | TypedDict for selective per-tool stopping |
| `ToolsToFinalOutputResult` | `agents.agent` | Dataclass returned by a custom `ToolsToFinalOutputFunction` |
| `Reasoning` | `openai.types.shared` | Controls reasoning effort for GPT-5 models |

> **Important:** `StopAtTools` and `ToolsToFinalOutputResult` are imported from `agents.agent` — **not** from the top-level `agents` package. They are not re-exported at the top level. Importing them from `agents` will raise an `ImportError`.

In [ ]:
from typing import Any

from openai.types.shared import Reasoning

from agents import (
    Agent,
    FunctionToolResult,
    ModelSettings,
    RunContextWrapper,
    Runner,
    function_tool,
)
from agents.agent import StopAtTools, ToolsToFinalOutputResult

## Cell 5 — What is `tool_use_behavior`?

### The question this parameter answers

When you give an agent tools, the LLM decides whether to call one. Once a tool runs and produces a result, the SDK faces a fork: send the result back to the LLM for synthesis, or return the raw result directly as the final output?

`tool_use_behavior` is the `Agent` field that controls this fork. It answers: *"After a tool call completes, what do we do with the result?"*

### How it relates to `tool_choice` (Lecture 2.7)

- `tool_choice` controls **whether and which** tools get called.
- `tool_use_behavior` controls **what happens to the result** once a tool has been called.

They are independent knobs on the same `Agent`.

### Scope: FunctionTools only

`tool_use_behavior` applies **only** to function tools you define with `@function_tool`. Hosted tools — web search, file search, code interpreter — are always processed by the LLM regardless of this setting. This is stated explicitly in the SDK source:

> *"NOTE: This configuration is specific to FunctionTools. Hosted tools, such as file search, web search, etc. are always processed by the LLM."*

### The four valid values at a glance

| Value | LLM sees result? | Use when |
|---|---|---|
| `"run_llm_again"` | Yes — always | Conversational agents; you want a polished, synthesised response |
| `"stop_on_first_tool"` | No — raw output returned | Tool output IS the answer; pipeline stages needing low latency |
| `StopAtTools([...])` | Only for non-listed tools | Mix of fast-path and conversational tools in the same agent |
| Custom function | You decide | Routing logic that depends on the actual content of the tool output |

## Cell 6 — Define Three Demo Tools

We define three lightweight city-information tools to use as props across all four demos. Each returns a hard-coded string so we can focus entirely on what `tool_use_behavior` does with the result — not on the tools themselves.

The `@function_tool` decorator:
- Reads the function's type annotations to build a JSON schema for the LLM
- Uses the docstring as the tool description the LLM sees
- Wraps the function so the SDK can invoke it during a run

Section 3 covers `@function_tool` in depth. For now, treat these as simple callable tools.

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Returns the current weather for a city."""
    print("[Tool Call] get_weather")
    return f"The weather in {city} is sunny and 24°C."


@function_tool
def get_population(city: str) -> str:
    """Returns the approximate population of a city."""
    print("[Tool Call] get_population")
    return f"The population of {city} is approximately 2.1 million."


@function_tool
def get_timezone(city: str) -> str:
    """Returns the timezone of a city."""
    print("[Tool Call] get_timezone")
    return f"{city} is in the UTC+5:30 timezone."

## Cell 7 — `"run_llm_again"` (the default)

With `"run_llm_again"` the SDK sends the tool result back to the LLM, which produces a natural language final response. This is the default — you do not need to set it explicitly, but we do so here for clarity.

**What to watch:**
- `final_output` is a polished sentence, not the raw string the tool returned.
- `new_items` count will be **3**: tool call item, tool output item, and the final LLM response item.

**When to use:**
- Conversational agents where a human-friendly response matters.
- When the model needs to synthesise or interpret the tool result before replying.

**Trade-off:** Requires a second LLM call — more tokens, slightly more latency.

In [ ]:
agent_default = Agent(
    name="Default Agent",
    instructions=(
        "You are a helpful assistant. "
        "Use tools to answer questions about cities."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_weather, get_population],
    tool_use_behavior="run_llm_again",
)

result = await Runner.run(
    agent_default,
    "What is the weather in Mumbai?",
)

print("run_llm_again:")
print(result.final_output)
print("Items:", len(result.new_items))
for item in result.new_items:
  print(item)

[Tool Call] get_weather
run_llm_again:
Mumbai: sunny, 24°C.
Items: 3
ToolCallItem(agent=Agent(name='Default Agent', handoff_description=None, tools=[FunctionTool(name='get_weather', description='Returns the current weather for a city.', params_json_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7ae4f9a68230>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='get_population', description='Returns the approximate population of a city.', params_json_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_population_args', 'type': 'object', 'additiona

## Cell 8 — `"stop_on_first_tool"`

With `"stop_on_first_tool"` the SDK takes the raw return value of the first tool call and uses it as `final_output` directly. No second LLM call happens.

**What to watch:**
- `final_output` is the exact string returned by the tool — unpolished, no synthesis.
- `new_items` count will be **2** (tool call + tool output, no final LLM message).

**Why we add `tool_choice="required"`:**  
Without it, the LLM might answer in plain text and never call the tool. `stop_on_first_tool` only fires if a tool is actually called, so we force one.

**When to use:**
- Pipeline stages where you need the raw data, not a human-friendly sentence about it.
- Latency-critical paths where the second LLM call is wasteful.

In [ ]:
agent_stop = Agent(
    name="Stop Agent",
    instructions=(
        "You are a helpful assistant. "
        "Use tools to answer questions about cities."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        tool_choice="required",
    ),
    tools=[get_weather, get_population],
    tool_use_behavior="stop_on_first_tool",
)

result = await Runner.run(
    agent_stop,
    "What is the weather in Mumbai?",
)

print("stop_on_first_tool:")
print(result.final_output)
print("Items:", len(result.new_items))

[Tool Call] get_weather
stop_on_first_tool:
The weather in Mumbai is sunny and 24°C.
Items: 2


## Cell 9 — `StopAtTools` — Selective Stopping

`StopAtTools` is the middle ground. You supply a list of tool names. If any of those tools is called during a turn, the SDK stops and returns that tool's raw output as `final_output`. Tools **not** in the list still route back through the LLM.

**Import note:** `StopAtTools` comes from `agents.agent`, not the top-level `agents` package.

**What to watch in the two queries below:**
- **Weather query:** `get_weather` is listed in `stop_at_tool_names` → raw string returned directly.
- **Population query:** `get_population` is **not** listed → LLM synthesises a response.

**When to use:**
- When you want per-tool control without writing a custom function.
- Agents that mix data-extraction tools (fast-path) and reasoning tools (need LLM synthesis).

In [ ]:
agent_stop_at = Agent(
    name="StopAtTools Agent",
    instructions=(
        "You are a helpful assistant. "
        "Use tools to answer questions about cities."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_weather, get_population, get_timezone],
    tool_use_behavior=StopAtTools(
        stop_at_tool_names=["get_weather"]
    ),
)

# get_weather is in the stop list → raw output returned directly
result_weather = await Runner.run(
    agent_stop_at,
    "What is the weather in Chennai?",
)
print("StopAtTools — weather query (fast-path):")
print(result_weather.final_output + "\n")

# get_population is NOT in the stop list → LLM synthesises
result_pop = await Runner.run(
    agent_stop_at,
    "What is the population of Chennai?",
)
print("StopAtTools — population query (LLM synthesis):")
print(result_pop.final_output)

[Tool Call] get_weather
StopAtTools — weather query (fast-path):
The weather in Chennai is sunny and 24°C.

[Tool Call] get_population
StopAtTools — population query (LLM synthesis):
Chennai has an approximate population of 2.1 million.


## Cell 10 — Custom `ToolsToFinalOutputFunction`

For complete programmatic control, pass a callable as `tool_use_behavior`. The function receives:

| Parameter | Type | Contents |
|---|---|---|
| `context` | `RunContextWrapper[Any]` | The run context for this turn |
| `results` | `list[FunctionToolResult]` | **All** tool results from the current turn |

It must return a `ToolsToFinalOutputResult` (imported from `agents.agent`) with two fields:

| Field | Type | Meaning |
|---|---|---|
| `is_final_output` | `bool` | `True` → run stops, `final_output` is returned. `False` → results go back to LLM. |
| `final_output` | `Any \| None` | The value to return when `is_final_output=True`. Set to `None` when `False`. |

**Access the tool's Python return value via `result.output`.**

**Both sync and async callables are supported.** We define an `async` function here, but a regular `def` works equally well.

In this example we check whether the tool output contains the word `"sunny"`. If it does, we add our own message and return it as the final output. Otherwise we pass the result back to the LLM.

In [ ]:
async def custom_behavior(
    context: RunContextWrapper[Any],
    results: list[FunctionToolResult],
) -> ToolsToFinalOutputResult:
    """Stop and augment the output if weather is sunny; otherwise let the LLM handle it."""
    for result in results:
        if isinstance(result.output, str) and "sunny" in result.output:
            return ToolsToFinalOutputResult(
                is_final_output=True,
                final_output=(
                    f"Great news! {result.output} "
                    f"Perfect weather for outdoor activities."
                ),
            )
    return ToolsToFinalOutputResult(
        is_final_output=False,
        final_output=None,
    )


agent_custom = Agent(
    name="Custom Behavior Agent",
    instructions=(
        "You are a helpful assistant. "
        "Use tools to answer questions about city weather."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_weather, get_population],
    tool_use_behavior=custom_behavior,
)

result = await Runner.run(
    agent_custom,
    "What is the weather in Sydney?",
)

print("Custom behavior:")
print(result.final_output)

[Tool Call] get_weather
Custom behavior:
Great news! The weather in Sydney is sunny and 24°C. Perfect weather for outdoor activities.


## Cell 11 — Decision Guide

Here is a practical decision table for choosing the right `tool_use_behavior` value.

| Value | LLM processes result? | Reach for it when… |
|---|---|---|
| `"run_llm_again"` | Yes — always | You want a polished, natural language response. This is the default and covers most conversational agents. |
| `"stop_on_first_tool"` | No — raw output returned | The tool output IS the answer. You are building a pipeline stage that just needs the data. |
| `StopAtTools([...])` | Only for non-listed tools | You have a mix: some tools should fast-path, others need LLM synthesis. No custom code required. |
| Custom function | You decide per call | Your routing logic depends on the **content** of the tool output, not just which tool was called. |

**Rule of thumb:** Start with `"run_llm_again"`. Switch to `"stop_on_first_tool"` when you need the raw data and lower latency. Use `StopAtTools` for per-tool control without writing a function. Reach for a custom function only when you need content-aware routing.